In [1]:
import os
import sys
import shutil
import subprocess
import numpy as np
import pandas as pd
from coords import *
from shapefiles import *
from peaksdata import *

# Data

Download prominence and isolation lists from Andrew Kirmse project:
https://github.com/akirmse/mountains

In [11]:
# path to prominence and isolation files
prominenceDB = '../raw_data/prominence-p100.txt'
isolationDB  = '../raw_data/alliso-sorted.txt'

if not os.path.exists(prominenceDB) or not os.path.exists(isolationDB):
    print('ERROR: peak databases not found!')

In [2]:
# region shapefiles
regionShapesDir = '../raw_data/regionShapes'
regionShapes = [f for f in os.listdir(regionShapesDir) if f.endswith('.shp')]

# Filter and unify prominence and isolation peak lists

In [3]:
regionPeaksDir = '../regionPeaks'

if not os.path.exists(regionPeaksDir):
    os.makedirs(regionPeaksDir)

In [33]:
# process each region to filter the database peaks that are inside
# since this process takes a long time, we provide the functions as standalone scripts for batch processing
for region in regionShapes:
    print(region)
    shp = os.path.join(regionShapesDir, region)

    print(' 1/3 Filtering prominence DB...')
    subprocess.run([sys.executable, 'filterPoints.py', shp, '../raw_data/prominence-p100.txt', 'prom.txt'])

    print(' 2/3 Filtering isolation DB...')
    subprocess.run([sys.executable, 'filterPoints.py', shp, '../raw_data/alliso-sorted.txt', 'isol.txt'])

    # merge lists
    print(' 3/3 Merging lists...')
    subprocess.run([sys.executable, 'mergePeaklists.py', 'isol.txt', 'prom.txt', 'tmppeaks.csv', '--deleteOriginals'])

    # move results to output dir
    shutil.move("tmppeaks.csv", os.path.join(regionPeaksDir, region.replace(".shp", ".csv")))

print('done!')

tienshan.shp
 1/3 Filtering prominence DB...
Found 48754 peaks
 2/3 Filtering isolation DB...
Found 26753 peaks
 3/3 Merging lists...
Reading isolations
Merging isolation and prominence lists
Processed 48754 peaks, low isolation on 29205
sierra.shp
 1/3 Filtering prominence DB...
Found 13722 peaks
 2/3 Filtering isolation DB...
Found 5996 peaks
 3/3 Merging lists...
Reading isolations
Merging isolation and prominence lists
Processed 13722 peaks, low isolation on 8622
taurus.shp
 1/3 Filtering prominence DB...
Found 18442 peaks
 2/3 Filtering isolation DB...
Found 13945 peaks
 3/3 Merging lists...
Reading isolations
Merging isolation and prominence lists
Processed 18442 peaks, low isolation on 9183
andes_peru.shp
 1/3 Filtering prominence DB...
Found 35081 peaks
 2/3 Filtering isolation DB...
Found 18118 peaks
 3/3 Merging lists...
Reading isolations
Merging isolation and prominence lists
Processed 35081 peaks, low isolation on 19116
alps.shp
 1/3 Filtering prominence DB...
Found 31625 

# Compute statistics 

In [4]:
regionStatsDir = '../regionStats'

if not os.path.exists(regionStatsDir):
    os.makedirs(regionStatsDir)

In [5]:
# statistics disk radius
diskRadius = 30

In [6]:
writeFeatures = ['elevation', 'elevRel', 'prominence', 'promRel', 
                 'dominance', 'domGroup', 'relevance',
                 'isolation', 'isolDir', 'saddleDist', 'saddleDir']

def writeHeaderToFile(fout, distributions):
    fout.write('lat,lon,peaks')
    for feat in writeFeatures:
        if feat in ['elevation', 'prominence', 'isolDir', 'saddleDir']:
            for val in distributions[feat]['bins'][:-1]:
                fout.write(',%s_%d' % (feat, int(val)))
        elif feat == 'domGroup':
            for val in distributions[feat]['bins'][:-1]:
                fout.write(',%s_%.2f' % (feat, 100*val))
        else:
            for val in distributions[feat]['bins'][:-1]:
                fout.write(',%s_%.2f' % (feat, val))
    fout.write('\n')

def writeLocationStatsToFile(fout, lat, lon, npeaks, distributions):
    fout.write('%.4f,%.4f,%d'%(lat, lon, npeaks))
    for feat in writeFeatures:
        for val in distributions[feat]['hist']:
            fout.write(',%d' % val)
    fout.write('\n')

In [7]:
# process each region (note: it takes a long time!)
for region in regionShapes:
    
    # sample stats locations inside polygon, separated at least 1/2 radius distance
    sampleLocations = sampleShapefileLocations(os.path.join(regionShapesDir, region), diskRadius)

    # region peaks DB
    df = pd.read_csv(os.path.join(regionPeaksDir, region.replace('.shp', '.csv')))
    df = addExtraColumns(df)
    
    # normalize distance columns
    df['isolation']  /= diskRadius
    df['saddleDist'] /= diskRadius
    
    # results file
    fout = open(os.path.join(regionStatsDir, region.replace('.shp', '.csv')), 'w')
    headerWritten = False
    
    # compute statistics
    for di,diskCenter in enumerate(sampleLocations):
        
        # filter peaks in disk using haversine distance
        peaks = filterPeaksHaversineDist(df, diskCenter, diskRadius)
                
        # skip if not enough peaks
        if peaks.shape[0] < 20:
            continue
        
        # compute statistics
        # diskRadius = 1   to have isolation/saddle dist histograms axis from 0 to 1, note we normalized distances before
        # detailed = False for the classification histograms, for synthesis we double the number of bins
        distributions = computeDistributions(peaks, diskRadius=1.0, detailed=False)
        
        # write dataset headers if first location
        if not headerWritten:
            writeHeaderToFile(fout, distributions)
            headerWritten = True
        
        # write data line
        writeLocationStatsToFile(fout, diskCenter[0], diskCenter[1], peaks.shape[0], distributions)
        
        print('%s: %3d/%3d samples'%(region, di+1, len(sampleLocations)), end='\r' if di+1 < len(sampleLocations) else '\n')
        
    fout.close()

print('done!')

tienshan.shp: 384/384 samples
sierra.shp:  40/ 40 samples
taurus.shp: 127/127 samples
andes_peru.shp: 207/207 samples
alps.shp: 162/162 samples
roraima.shp: 454/454 samples
balkans.shp: 242/242 samples
highlands.shp:  42/ 42 samples
hindukush.shp: 127/127 samples
southafrica.shp: 331/331 samples
zagros.shp: 264/264 samples
patagonia.shp: 173/173 samples
colorado.shp: 153/153 samples
norway.shp: 266/266 samples
alaska.shp: 206/206 samples
andes_bolivia.shp: 106/106 samples
siberia.shp: 1011/1011 samples
cascades.shp:  97/ 97 samples
urals.shp: 164/164 samples
pamir.shp: 143/143 samples
andes_colombia.shp: 257/257 samples
turkey.shp: 235/235 samples
pyrenees.shp:  37/ 37 samples
rockies.shp: 199/199 samples
papua.shp: 240/240 samples
ethiopia.shp: 594/594 samples
japan.shp: 360/360 samples
apennines.shp:  51/ 51 samples
sahara.shp: 170/170 samples
newzealand.shp:  94/ 94 samples
nevada.shp: 207/207 samples
himalaya.shp: 240/240 samples
carpathian.shp: 118/118 samples
laos.shp: 492/492 sa

# Create dataset

In [8]:
# file where the dataset will be stored
fileDataset = '../regions_%dkm.csv' % (int(diskRadius))

# regions to put in the dataset (for example, we could omit certain regions, we can also do it later in classifier)
datasetRegions = [f for f in os.listdir(regionStatsDir) if f.endswith('.csv')]

# if True: write frequencies, otherwise keep histogram counts
# we observed that frequencies work better in the classifier
normalize = True

In [10]:
alldf = []
for file in datasetRegions:
    
    # name
    terrainName = file.split('.')[0]
    
    # read dataframe 
    df = pd.read_csv(os.path.join(regionStatsDir, file))
    
    # keep number of peaks
    npeaks = df['peaks'].values
    
    # drop lat, lon, npeaks
    df.drop(['lat', 'lon', 'peaks'], axis=1, inplace=True)
    
    # normalize histogram columns?
    if normalize:
        for c in df.columns:
            df[c] = df[c].astype(np.float32)/npeaks
    
    # add terrain name column
    df.insert(0, 'terrain', terrainName)
    alldf.append(df)
    
    print('%4d %s' % (df.shape[0], terrainName))
    
alldf = pd.concat(alldf, ignore_index=True)
alldf.to_csv(fileDataset, float_format='%.4f', index=False)

/tmp/ipykernel_22796/2914976056.py:14: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(['lat', 'lon', 'peaks'], axis=1, inplace=True)
/tmp/ipykernel_22796/2914976056.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.insert(0, 'terrain', terrainName)
/tmp/ipykernel_22796/2914976056.py:14: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(['lat', 'lon', 'peaks'], axis=1, inplace=True)


 264 zagros
 127 hindukush


/tmp/ipykernel_22796/2914976056.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.insert(0, 'terrain', terrainName)
/tmp/ipykernel_22796/2914976056.py:14: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(['lat', 'lon', 'peaks'], axis=1, inplace=True)
/tmp/ipykernel_22796/2914976056.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.insert(0, 't

 143 pamir
  42 highlands
  97 cascades


/tmp/ipykernel_22796/2914976056.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.insert(0, 'terrain', terrainName)
/tmp/ipykernel_22796/2914976056.py:14: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(['lat', 'lon', 'peaks'], axis=1, inplace=True)
/tmp/ipykernel_22796/2914976056.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.insert(0, 't

 266 norway
 207 nevada
 368 tienshan


... (30 outputs left out)

/tmp/ipykernel_22796/2914976056.py:14: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(['lat', 'lon', 'peaks'], axis=1, inplace=True)
/tmp/ipykernel_22796/2914976056.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.insert(0, 'terrain', terrainName)
/tmp/ipykernel_22796/2914976056.py:14: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(['lat', 'lon', 'peaks'], axis=1, inplace=True)
/tmp/ipykernel_22796/

 221 altai
 357 japan


/tmp/ipykernel_22796/2914976056.py:14: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(['lat', 'lon', 'peaks'], axis=1, inplace=True)
/tmp/ipykernel_22796/2914976056.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.insert(0, 'terrain', terrainName)
/tmp/ipykernel_22796/2914976056.py:14: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(['lat', 'lon', 'peaks'], axis=1, inplace=True)
/tmp/ipykernel_22796/

 122 caucasus
 153 colorado
